# VendorCenter AI — Fine-Tuning Notebook (v2)

**Model**: `unsloth/Qwen2.5-3B-Instruct` (QLoRA 4-bit)
**Output**: GGUF Q4_K_M for llama.cpp on HuggingFace Spaces
**Data**: ~1400 examples, 14 intents, bracketed output format
**Chat template**: ChatML (`<|im_start|>`)

| Step | Cell |
|------|------|
| Install deps | 1 |
| Decode training data | 2 |
| Load & validate | 3 |
| Format ChatML + split | 4 |
| Load model (4-bit) | 5 |
| Add LoRA adapters | 6 |
| Configure trainer | 7 |
| Train | 8 |
| Loss curves | 9 |
| Eval on held-out set | 10 |
| Manual test suite | 11 |
| Save LoRA | 12 |
| Export GGUF Q4_K_M | 13 |
| Push to HuggingFace | 14 |

> **Runtime**: T4 GPU (free tier). Training takes ~30-50 min.

In [ ]:
%%capture
# Official unsloth Colab install — matches their latest notebooks
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
# Extra deps for eval plots
!pip install -q scikit-learn matplotlib

In [ ]:
# Cell 2: Transfer training data (base64-encoded gzip)
import base64, gzip, os

B64 = "..."

data = gzip.decompress(base64.b64decode(B64))
with open('vendorcenter_train.jsonl', 'wb') as f:
    f.write(data)
size_kb = os.path.getsize('vendorcenter_train.jsonl') / 1024
print(f'Training file written: {size_kb:.1f} KB')


In [ ]:
# Cell 3: Load, inspect & validate training data
import json, re
from collections import Counter

VALID_INTENTS = {'GREETING', 'SERVICE_SEARCH', 'RECOMMENDATION', 'BOOKING', 'MY_BOOKINGS',
                 'AVAILABLE_SERVICES', 'FAQ', 'COMPLAINT', 'RESCHEDULE', 'CANCEL_BOOKING',
                 'REFUND', 'VENDOR_INFO', 'LOCATION', 'UNKNOWN'}

def validate_bracketed(output_text):
    lines = output_text.strip().split('\n')
    if len(lines) < 2:
        return None, False
    tag_line = lines[0]
    message = '\n'.join(lines[1:]).strip()
    if not message:
        return None, False
    intent_match = re.search(r'\[INTENT:(\w+)\]', tag_line)
    if not intent_match:
        return None, False
    intent = intent_match.group(1)
    if intent not in VALID_INTENTS:
        return intent, False
    if not re.search(r'\[ACTION:\w+\]', tag_line):
        return intent, False
    if not re.search(r'\[CONFIDENCE:[\d.]+\]', tag_line):
        return intent, False
    return intent, True

examples = []
with open('vendorcenter_train.jsonl') as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

valid_count = 0
for ex in examples:
    intent, is_valid = validate_bracketed(ex['output'])
    ex['_intent'] = intent or 'UNKNOWN'
    if is_valid:
        valid_count += 1

print(f'Total examples: {len(examples)}')
print(f'Valid bracketed: {valid_count}/{len(examples)} ({valid_count/len(examples)*100:.1f}%)')

intent_dist = Counter(ex['_intent'] for ex in examples)
print(f'\nIntent distribution ({len(intent_dist)} intents):')
for intent, count in intent_dist.most_common():
    pct = count / len(examples) * 100
    print(f'  {intent:25s} {count:4d}  ({pct:.1f}%)')

print(f'\nSample input:  {examples[0]["input"][:100]}')
print(f'Sample output: {examples[0]["output"][:200]}')

In [ ]:
# Cell 4: Format into ChatML for Qwen2.5-Instruct + train/eval split
import random
random.seed(42)

EOS_TOKEN = "<|im_end|>"

def format_chatml(example):
    text = (
        f'<|im_start|>system\n'
        f'{example["instruction"]}<|im_end|>\n'
        f'<|im_start|>user\n'
        f'{example["input"]}<|im_end|>\n'
        f'<|im_start|>assistant\n'
        f'{example["output"]}<|im_end|>'
    )
    return {
        'text': text,
        'intent': example['_intent'],
        'raw_input': example['input'],
        'raw_output': example['output'],
    }

formatted = [format_chatml(ex) for ex in examples]

# Shuffle then split 90/10
random.shuffle(formatted)
split_idx = int(len(formatted) * 0.9)
train_formatted = formatted[:split_idx]
eval_formatted = formatted[split_idx:]

print(f'Train: {len(train_formatted)} | Eval: {len(eval_formatted)}')
print(f'\nSample (first 400 chars):\n{train_formatted[0]["text"][:400]}')

In [ ]:
# Cell 5: Load Qwen2.5-3B-Instruct with 4-bit quantization
from unsloth import FastLanguageModel
import torch

if not torch.cuda.is_available():
    raise RuntimeError("NO GPU! Go to Runtime > Change runtime type > T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

max_seq_length = 2048  # Official unsloth default for T4. Cell 6 pre-filters any outliers.
dtype = None           # Auto-detect (float16 for T4)
load_in_4bit = True    # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print(f'Model loaded. Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 6: Create HuggingFace Datasets — filter out any sequences > max_seq_length
from datasets import Dataset

# Pre-check: tokenize and drop any examples that exceed max_seq_length
# This prevents the fused CE loss from crashing on truncation mismatches
safe_train = []
safe_eval  = []
dropped = 0

for ex in train_formatted:
    tok_len = len(tokenizer.encode(ex['text'], add_special_tokens=False))
    if tok_len <= max_seq_length - 2:  # leave room for BOS/EOS
        safe_train.append({'text': ex['text']})
    else:
        dropped += 1

for ex in eval_formatted:
    tok_len = len(tokenizer.encode(ex['text'], add_special_tokens=False))
    if tok_len <= max_seq_length - 2:
        safe_eval.append({'text': ex['text']})
    else:
        dropped += 1

print(f'Dropped {dropped} examples exceeding {max_seq_length} tokens')
train_dataset = Dataset.from_list(safe_train)
eval_dataset  = Dataset.from_list(safe_eval)
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# Cell 7: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,      # r = alpha for stable LoRA scaling
    lora_dropout=0,     # 0 is optimized by unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
# Cell 8: Configure SFTTrainer — T4 optimized
from trl import SFTConfig, SFTTrainer
import torch, os

torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # effective batch = 16
        warmup_steps=30,
        num_train_epochs=8,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="vendorcenter-lora",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)
print('Trainer ready.')

In [ ]:
# Cell 9: Train!
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f'\nTraining complete in {elapsed/60:.1f} minutes')
print(f'Final loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# Cell 10: Training stats + loss curve
import matplotlib.pyplot as plt

# Memory stats
used  = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
total = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
print(f"Peak reserved memory: {used} GB / {total} GB ({round(used/total*100, 1)}%)")

log_history = trainer.state.log_history
train_steps  = [h['step'] for h in log_history if 'loss' in h]
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_steps   = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses  = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', alpha=0.8)
if eval_losses:
    plt.plot(eval_steps, eval_losses, 'ro-', label='Eval Loss', markersize=8)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('VendorCenter Fine-Tuning — Loss Curve')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

print(f'Train loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Eval loss:  {eval_losses[0]:.4f} -> {eval_losses[-1]:.4f}')

In [ ]:
# Cell 11: Evaluation on held-out set
import re

EVAL_SYSTEM_PROMPT = """You are VendorCenter AI, a helpful assistant for a local services marketplace in India.
Given the user's message, respond with bracketed tags on the first line followed by a friendly message.

Format: [INTENT:X] [SERVICE:Y] [ACTION:Z] [CONFIDENCE:N.N]
Your helpful message here.

Intents: GREETING, SERVICE_SEARCH, RECOMMENDATION, BOOKING, MY_BOOKINGS, AVAILABLE_SERVICES, FAQ, COMPLAINT, RESCHEDULE, CANCEL_BOOKING, REFUND, VENDOR_INFO, LOCATION, UNKNOWN
Services: AC Repair, Appliance Repair, Catering, Cleaning, Carpentry, Computer Repair, Electrical, Fitness, Mobile Repair, Moving, Painting, Pest Control, Photography, Plumbing, Salon, Tutoring
Actions: SHOW_RESULTS, GET_RECOMMENDATIONS, BOOK_SERVICE, SHOW_MY_BOOKINGS, SHOW_CATEGORIES, ASK_LOCATION, ASK_DETAILS, NAVIGATE, NONE

Support English, Hinglish, and Marathi. Be warm, use emojis sparingly. Never invent vendor data."""

FastLanguageModel.for_inference(model)  # 2x faster inference

def run_inference(user_msg):
    prompt = (
        f'<|im_start|>system\n{EVAL_SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_msg}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    result = tokenizer.batch_decode(output[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return result

def parse_bracketed(text):
    lines = text.strip().split('\n')
    if len(lines) < 2:
        return None
    tag_line = lines[0]
    intent_m = re.search(r'\[INTENT:(\w+)\]', tag_line)
    action_m = re.search(r'\[ACTION:(\w+)\]', tag_line)
    conf_m   = re.search(r'\[CONFIDENCE:([\d.]+)\]', tag_line)
    if not intent_m or not action_m or not conf_m:
        return None
    return {
        'intent': intent_m.group(1),
        'action': action_m.group(1),
        'confidence': float(conf_m.group(1)),
        'message': '\n'.join(lines[1:]).strip(),
    }

# Evaluate
format_valid = 0
intent_correct = 0
total = len(eval_formatted)
intent_results = {}

print(f'Evaluating on {total} held-out examples...')
for i, ex in enumerate(eval_formatted):
    expected = ex['intent']
    response = run_inference(ex['raw_input'])
    if expected not in intent_results:
        intent_results[expected] = [0, 0]
    intent_results[expected][1] += 1
    parsed = parse_bracketed(response)
    if parsed:
        format_valid += 1
        if parsed['intent'] == expected:
            intent_correct += 1
            intent_results[expected][0] += 1
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{total} — Valid: {format_valid}/{i+1}, Intent: {intent_correct}/{i+1}')

print(f'\n{"="*50}')
print(f'RESULTS ({total} examples)')
print(f'{"="*50}')
print(f'Format validity:  {format_valid}/{total} ({format_valid/total*100:.1f}%)')
print(f'Intent accuracy:  {intent_correct}/{total} ({intent_correct/total*100:.1f}%)')
print(f'\nPer-intent:')
for intent in sorted(intent_results.keys()):
    correct, count = intent_results[intent]
    acc = correct / count * 100 if count > 0 else 0
    s = 'OK' if acc >= 80 else 'WARN' if acc >= 50 else 'FAIL'
    print(f'  [{s:4s}] {intent:25s} {correct:3d}/{count:3d} ({acc:.0f}%)')

In [ ]:
# Cell 12: Manual test suite
test_cases = [
    ('I need a plumber near me', 'SERVICE_SEARCH'),
    ('How do I book a service?', 'FAQ'),
    ('hello', 'GREETING'),
    ('Show my bookings', 'MY_BOOKINGS'),
    ('What services are available?', 'AVAILABLE_SERVICES'),
    ('The vendor did a terrible job', 'COMPLAINT'),
    ('I need to reschedule my booking', 'RESCHEDULE'),
    ('Cancel my booking', 'CANCEL_BOOKING'),
    ('Where is my refund?', 'REFUND'),
    ('Show me vendor reviews', 'VENDOR_INFO'),
    ('I am in Pune', 'LOCATION'),
    ('Tell me a joke', 'UNKNOWN'),
    ('AC kharab ho gaya hai', 'SERVICE_SEARCH'),
    ('mujhe cancel karna hai', 'CANCEL_BOOKING'),
    ('complaint raise karna hai', 'COMPLAINT'),
    ('booking ka status kya hai?', 'MY_BOOKINGS'),
    ('URGENT pipe burst help!!!!', 'SERVICE_SEARCH'),
    ('worst service ever', 'COMPLAINT'),
    ('need electrician asap', 'SERVICE_SEARCH'),
]

print(f'Running {len(test_cases)} test cases...\n')
passed = 0
for user_msg, expected_intent in test_cases:
    response = run_inference(user_msg)
    parsed = parse_bracketed(response)
    if parsed:
        actual = parsed['intent']
        match = actual == expected_intent
        if match:
            passed += 1
        icon = 'PASS' if match else 'FAIL'
        print(f'[{icon}] "{user_msg[:45]:<45s}" expected={expected_intent:<20s} got={actual}')
        if not match:
            print(f'       Response: {response[:120]}')
    else:
        print(f'[FAIL] "{user_msg[:45]:<45s}" expected={expected_intent:<20s} got=INVALID_FORMAT')
        print(f'       Raw: {response[:120]}')

print(f'\nPassed: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
# Cell 13: Save LoRA adapter
model.save_pretrained("vendorcenter-lora-final")
tokenizer.save_pretrained("vendorcenter-lora-final")
print('LoRA adapter saved to vendorcenter-lora-final/')

In [ ]:
# Cell 14: Merge LoRA + export GGUF Q4_K_M
model.save_pretrained_gguf(
    "vendorcenter-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

import os
# Unsloth appends _gguf to the output dir name
gguf_dir = "vendorcenter-gguf"
if os.path.exists("vendorcenter-gguf_gguf"):
    gguf_dir = "vendorcenter-gguf_gguf"

if os.path.exists(gguf_dir):
    found = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]
    if found:
        for f in found:
            fpath = os.path.join(gguf_dir, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"{f}: {size_mb:.2f} MB")
    else:
        print("No .gguf files found in directory.")
else:
    print(f"Directory {gguf_dir} not found.")

In [ ]:
# Cell 15: Push GGUF to HuggingFace Hub
# check model here https://huggingface.co/timesprimeaj/vendorcenter-assistant-qwen25-gguf
HF_TOKEN = 'token_goes_here'  # Replace with your HuggingFace token
HF_REPO  = 'timesprimeaj/vendorcenter-assistant-qwen25-gguf'

from huggingface_hub import HfApi
import os

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, exist_ok=True, repo_type='model')
print(f'Repo: https://huggingface.co/{HF_REPO}')

# Check both possible unsloth output names
gguf_dir = 'vendorcenter-gguf'
if os.path.exists('vendorcenter-gguf_gguf'):
    gguf_dir = 'vendorcenter-gguf_gguf'

if not os.path.exists(gguf_dir):
    raise FileNotFoundError(f'No GGUF output directory found at {gguf_dir}!')

gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith('.gguf')]
if not gguf_files:
    raise ValueError(f'No .gguf files in {gguf_dir}/. Please check if cell 14 succeeded.')

for f in gguf_files:
    fpath = os.path.join(gguf_dir, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f'Uploading {f} ({size_mb:.2f} MB)...')
    api.upload_file(path_or_fileobj=fpath, path_in_repo=f, repo_id=HF_REPO)
    print(f'  Done')

print(f'\nModel live at: https://huggingface.co/{HF_REPO}')

## Next Steps

1. **Deploy on HF Spaces**: Use `model/hf-space/Dockerfile` (uses `--chat-template chatml`)
2. **Set backend env**: `SELF_HOSTED_LLM_URL=https://YOUR-SPACE.hf.space` on Railway
3. **Test**: Provider chain tries your model first, then Groq, then Gemini
4. **Model repo**: `timesprimeaj/vendorcenter-assistant-qwen25-gguf`

### Improving the Model
- Run `npx tsx model/scripts/generate-training-data.ts` to regenerate training data
- Add real user queries from `ai_query_logs` to training data
- If eval loss plateaus: increase LoRA rank (32/64) or epochs (7-10)
- If format validity < 90%: add more format-diverse examples
- Monitor per-intent accuracy — low scores = more examples needed